In [ ]:
from notebook.services.config import ConfigManager
cm = ConfigManager()
cm.update('livereveal', {
        'width': 1920,
        'height': 1080,
        'scroll': True,
})

# Week 08: Monday, AST 5011: Astrophysical Systems

## The Galaxy Population

### Michael Coughlin <cough052@umn.edu>

With contributions from Frank van den Bosch (Yale) and Benedikt Diemer (UMD).

Textbook reference: *Introduction to Galaxy Formation and Evolution* by Cimatti, Fraternali, and Nipoti (CFN).

In [ ]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import routines
import os

from colossus.cosmology import cosmology

# Plotting settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# Automatically reload code imported from changed python files
%reload_ext autoreload
%autoreload 2

# Cosmological Tools

Throughout the course, we will need cosmological calculations such as distances. We use the [Colossus](https://bdiemer.bitbucket.io/colossus/) code for this purpose.

In [ ]:
cosmo = cosmology.setCosmology('planck18')
print(cosmo)

The Colossus cosmology module provides standard calculations: densities, distances, and times. Redshift is always the argument.

In [ ]:
print('Age of the universe: %.2f Gyr' % cosmo.age(0.0))

# Cosmological distances
z = np.array([0.0, 1.0, 2.0, 3.0])
print('Angular diameter distances (Mpc/h):', cosmo.angularDiameterDistance(z))

### Cosmological Distances

Different distance measures diverge at high redshift. The luminosity distance $d_L$ grows faster than comoving distance $d_C$, while the angular diameter distance $d_A$ turns over around $z \sim 1$.

In [ ]:
z_min = -2.0
z_max = 2.0
z = 10**np.arange(z_min, z_max, 0.02)

dC = cosmo.comovingDistance(0.0, z)
dL = cosmo.luminosityDistance(z)
dA = cosmo.angularDiameterDistance(z)

plt.figure(figsize = (3.5, 3.5))
plt.loglog()
plt.xlabel(r'$z$')
plt.ylabel(r'${\rm Mpc}/h$')
plt.xlim(10**z_min, 10**z_max)
plt.plot(z, dC, label = r'$d_{\rm C}$')
plt.plot(z, dL, label = r'$d_{\rm L}$')
plt.plot(z, dA, label = r'$d_{\rm A}$')
plt.legend()
plt.show()

# The SDSS Galaxy Sample

The [Sloan Digital Sky Survey](https://www.sdss.org/) (SDSS) provides the largest spectroscopic galaxy sample, with photometry in five bands ($ugriz$) and redshifts for hundreds of thousands of galaxies. We load a pre-processed version of the SDSS DR8 spectroscopic galaxy sample.

In [ ]:
from routines import loadSdssSpecSampleExtra

data = loadSdssSpecSampleExtra()

print(f"Found {len(data)} galaxies")

### Distribution on the Sky

Let's plot the SDSS footprint using a Hammer projection.

In [ ]:
ra_rad = data['ra'] * np.pi / 180.0
dec_rad = data['dec'] * np.pi / 180.0

apache_point_dec = 32.78
dec_slice_min = -2.0
dec_slice_max = 10.0
ra_array = np.linspace(-np.pi, np.pi, 100)
ones_array = np.ones((100), float)

fig = plt.figure(figsize = (10.0, 10.0))
ax = fig.add_subplot(111, projection = 'hammer')
plt.scatter(ra_rad - np.pi, dec_rad, marker = '.', s = 1.0, linewidths = 0, alpha = 0.1)
plt.plot(ra_array, ones_array * apache_point_dec * np.pi / 180.0, '--', color = 'gray')
plt.plot(ra_array, ones_array * dec_slice_min * np.pi / 180.0, ':', color = 'gray')
plt.plot(ra_array, ones_array * dec_slice_max * np.pi / 180.0, ':', color = 'gray')
plt.show()

### Distribution in Redshift

The SDSS spectroscopic sample covers roughly $0 < z < 0.3$, with a tail to higher redshifts.

In [ ]:
plt.figure(figsize = (4.0, 4.0))
plt.xlim(0.0, 0.7)
plt.xlabel(r'$z$')
plt.ylabel(r'$N_{\rm gal}$')
plt.hist(data['z'], bins = 30, log = True)
plt.show()

plt.figure(figsize = (4.0, 4.0))
plt.xlim(0.0, 0.4)
plt.xlabel(r'$z$')
plt.ylabel(r'$N_{\rm gal}$')
plt.hist(data['z'], bins = 30, log = False, range = [0, 0.4])
plt.show()

### Large-Scale Structure

A redshift cone plot reveals filaments, voids, and walls in the galaxy distribution. The "Sloan Great Wall" is visible as a massive filament.

In [ ]:
z_max = 0.16

mask = (data['z'] <= z_max) & (data['dec'] >= dec_slice_min) & (data['dec'] <= dec_slice_max)
ra_rad_cone = data['ra'][mask] * np.pi / 180.0
z_cone = data['z'][mask]

fig = plt.figure(figsize = (7, 7))
ax = fig.add_subplot(111, projection = 'polar')
ax.set_rlabel_position(-88.0)
ax.scatter(ra_rad_cone, z_cone, marker = '.', s = 1.2, linewidths = 0, rasterized = True)
ax.set_rmax(z_max)
ax.grid(ls = ':')
label_position = ax.get_rlabel_position()
ax.text(np.radians(label_position + 16), ax.get_rmax() * 0.5, r'$\mathrm{Redshift}$', rotation = label_position, ha = 'center', va = 'center')
plt.show()

### What Do SDSS Galaxies Look Like?

Let's plot a collage of SDSS galaxy images from a narrow distance range.

In [ ]:
n_rows = 5
n_cols = 5
n_pix = 70
scale = routines.sdss_pixel_scale
np.random.seed(2025)

dL = cosmo.luminosityDistance(data['z'])
idxs_valid = np.where((dL >= 200.0) & (dL <= 300.0))[0]
n_valid = len(idxs_valid)
idxs = idxs_valid[np.random.randint(0, n_valid - 1, n_rows * n_cols)]
d = data[idxs]

routines.imageCollage(d, n_rows, n_cols, n_pix = n_pix, scale = scale, panel_size = 2.0);

# Magnitudes, Extinction, and K-corrections

## SDSS Filter Bands

SDSS images the sky through five broadband filters ($u, g, r, i, z$), spanning from near-UV to near-IR.

In [ ]:
plt.figure(figsize = (6.0, 3.4))
plt.xlim(3000, 10500)
plt.ylim(0, 0.58)
plt.xlabel(r'$\lambda\ ({\rm \AA})$')
plt.ylabel(r'$S(\lambda)$')
for i, f, c, loc in zip([0, 1, 2, 3, 4], 'ugriz', 'bgrmk', [3500, 4650, 6150, 7500, 8750]):
    fn = os.path.join(routines.data_dir,'sdss/filter_%c.txt' % (f))
    filt = np.loadtxt(fn, unpack = True)
    if i == 0:
        label1 = r'$\mathrm{Without\ atmosphere}$'
        label2 = r'$\mathrm{With\ atmosphere}$'
    else:
        label1 = None
        label2 = None
    plt.plot(filt[0], filt[3], color = c, ls = '--', lw = 1.0, label = label1)
    plt.fill(filt[0], filt[2], ec = c, fc = c, alpha = 0.4, label = label2)
    plt.text(loc, 0.03, f, color = c, ha = 'center', va = 'center', fontsize = 14)
plt.legend()
plt.show()

### Apparent Magnitudes

SDSS provides several magnitude definitions: ``petroMag``, ``modelMag``, ``cmodelMag``, ``expMag``, ``deVMag``. Each measures flux within a different aperture or profile fit.

In [ ]:
plt.figure()
plt.gca().invert_xaxis()
plt.xlabel(r'$m_{\rm r}$')
plt.ylabel(r'$N_{\rm gal}$')
hist_args = dict(range = [13.0, 18.5], bins = 30)
plt.hist(data['petroMag_r'], histtype = 'stepfilled', label = r'$\mathrm{Petrosian}$', **hist_args)
plt.hist(data['cmodelMag_r'], histtype = 'step', label = r'$\mathrm{cModel}$', **hist_args)
plt.hist(data['modelMag_r'], histtype = 'step', alpha = 0.5, label = r'$\mathrm{Model}$', **hist_args)
plt.legend()
plt.show()

### The Petrosian Radius

Several of the magnitude definitions above reference the *Petrosian* radius. Unlike model-based radii (which depend on fitting a profile), the Petrosian radius is defined purely from the observed surface brightness profile. It is the radius $R_P$ at which the ratio $\eta$ of the local surface brightness to the mean enclosed surface brightness drops below a threshold:

$$\eta(R) = \frac{\bar{\Sigma}(0.8R,\ 1.25R)}{\bar{\Sigma}(0,\ R)} = 0.2$$

The Petrosian flux is then measured within $2R_P$, capturing a fixed fraction of the total light regardless of distance (in principle). This avoids the need to extrapolate a model profile to infinity. The ``petroR50`` and ``petroR90`` fields give the radii enclosing 50% and 90% of the Petrosian flux, respectively. Their ratio defines the *concentration* parameter, $c = R_{90}/R_{50}$, which we will use later as a morphological indicator.

### Dust Extinction

The Milky Way dust extinction for each galaxy and filter band is provided in the SDSS dataset. We must correct for this to get intrinsic magnitudes.

In [ ]:
hist_args = dict(range = [0.0, 0.6], bins = 30)

plt.figure()
plt.xlabel(r'$E_{\rm MW}$')
plt.ylabel(r'$N_{\rm gal}$')
plt.xlim(0.0, 0.6)
plt.hist(data['extinction_g'], histtype = 'stepfilled', label = r'$\mathrm{g}$', **hist_args)
plt.hist(data['extinction_r'], histtype = 'step', label = r'$\mathrm{r}$', **hist_args)
plt.legend()
plt.show()

f_absorbed = 1.0 - 10**(-0.4 * np.median(data['extinction_g']))
print('%.1f%% of g-band light is absorbed for the typical galaxy.' % (100.0 * f_absorbed))

### K-corrections

To compare galaxies at different redshifts, we need K-corrections to account for the redshifting of the spectrum through the filter bandpass. We use the fitting functions of Chilingarian et al. (2010).

### Galaxy Spectra and Color

To build physical intuition for why K-corrections matter, let's look at actual galaxy spectra alongside the SDSS filter bands. The spectra come from SDSS fibers (3" diameter) pointed at the galaxy centers. As stellar populations age, their spectra shift from blue to red and the 4000 Angstrom break becomes more prominent. When these spectra are redshifted, different parts of the spectral energy distribution (SED) pass through the filter bands, making K-corrections essential for accurate color measurements.

In [ ]:
color_bin_edges = [0.0, 0.3, 0.5, 0.7, 0.9, 1.1, 2.0]
np.random.seed(2025)

n_color_bins = len(color_bin_edges) - 1
dL_spec = cosmo.luminosityDistance(data['z']) / cosmo.h
gr_fib = data['fiberMag_g'] - data['fiberMag_r']
M_r_spec = data['modelMag_r'] - data['extinction_r'] - cosmo.distanceModulus(data['z'])

for i in range(n_color_bins):
    idxs = np.where((M_r_spec < -20.0) & (dL_spec <= 100.0)
                    & (gr_fib >= color_bin_edges[i]) & (gr_fib <= color_bin_edges[i + 1]))[0]
    if len(idxs) == 0:
        continue
    idx = np.random.choice(idxs, 1)[0]
    routines.imageAndSpectrum(data[idx], scale = 0.6)

In [ ]:
z = np.linspace(0.0, 0.6, 100)
gr = np.linspace(0.0, 1.6, 9)
cmap = plt.get_cmap('viridis')

fig, axes = plt.subplots(1, 2, figsize = (9.0, 4.0))

plt.sca(axes[0])
plt.text(0.1, 0.85, r'$g$', fontsize = 20, transform = plt.gca().transAxes)
for i in range(len(gr)):
    c = cmap(i / (len(gr) - 1))
    K = routines.kCorrection('g', z, 'g-r', gr[i])
    plt.plot(z, K, c = c, label = r'$g-r = %.1f$' % (gr[i]))
plt.legend(labelspacing = 0.05)

plt.sca(axes[1])
plt.text(0.1, 0.85, r'$r$', fontsize = 20, transform = plt.gca().transAxes)
for i in range(len(gr)):
    c = cmap(i / (len(gr) - 1))
    K = routines.kCorrection('r', z, 'g-r', gr[i])
    plt.plot(z, K, c = c)

for ax in axes:
    plt.sca(ax)
    plt.xlabel(r'$z$')
    plt.xlim(0.0, 0.6)
    plt.ylim(-3.0, 2.0)
axes[0].set_ylabel(r'$K(z)$')
axes[1].set_yticklabels([])
plt.show()

## In-Class Exercise: Absolute Magnitudes

### Goal
Convert apparent magnitudes to absolute magnitudes by applying the distance modulus and dust extinction corrections.

### Task

Using the SDSS galaxy sample:

1. Compute the distance modulus $\mathrm{DM} = 5 \log_{10}(d_L / 10\,\mathrm{pc})$ using Colossus.
2. Compute absolute magnitudes: $M_r = m_r - A_r - \mathrm{DM}$, where $A_r$ is the extinction.
3. Plot the histogram of $M_r$.
4. Print the median and the 0.5th/99.5th percentile range.

In [ ]:
# Compute luminosity distance in Mpc, then distance modulus
dL = cosmo.luminosityDistance(data['z']) / cosmo.h
dm = ... # FILL IN

# Compute absolute magnitude (ignoring K-corrections for now)
M_r = ... # FILL IN

# Compute percentiles
perc_low = ... # FILL IN
perc_high = ... # FILL IN
mag_range = perc_low - perc_high
print('99%% of galaxies lie between Mr = %.2f and %.2f, a range of %.1f magnitudes.' \
      % (perc_low, perc_high, mag_range))

# Plot
hist_args = dict(range = [-31.0, 9.0], bins = 30)
plt.figure()
plt.gca().invert_xaxis()
plt.xlabel(r'$M_{\rm r}$')
plt.ylabel(r'$N_{\rm gal}$')
plt.hist(M_r, histtype = 'stepfilled', log = True, **hist_args)
plt.axvline(perc_low, ls = '--', color = 'gray')
plt.axvline(perc_high, ls = '--', color = 'gray')
plt.show()

<details>
<summary>Solution</summary>

```python
dm = 5.0 * np.log10(dL * 1E6 / 10.0)
M_r = data['modelMag_r'] - data['extinction_r'] - dm
perc_low = np.percentile(M_r, 99.5)
perc_high = np.percentile(M_r, 0.5)
```

The result shows galaxies spanning roughly $M_r \approx -14$ to $-24$, a range of about 10 magnitudes or 4 orders of magnitude in luminosity. The distribution peaks around $M_r \approx -20$.

</details>

# Surface Brightness Profiles

## Sersic Profiles

The Sersic profile is a widely used parametric form for galaxy surface brightness:

$$\Sigma(R) = \Sigma_e \exp\left[-b_n \left(\left(\frac{R}{R_e}\right)^{1/n} - 1\right)\right]$$

where $n$ is the Sersic index, $R_e$ is the half-light (effective) radius, and $b_n \approx 1.9992n - 0.3271$.

Special cases:
- $n = 1$: exponential profile (disk galaxies)
- $n = 4$: de Vaucouleurs profile (elliptical galaxies)

## In-Class Exercise: Sersic Profiles

### Goal
Plot the Sersic surface brightness profile family to build intuition for how the index $n$ controls the shape.

### Task

For $n = 0.5, 1, 2, 4, 10$:

1. Compute $b_n = 1.9992n - 0.3271$.
2. Compute $\Sigma/\Sigma_e = \exp[-b_n((R/R_e)^{1/n} - 1)]$ on a log-spaced radial grid.
3. Also compute the surface brightness $\mu - \mu_e = 1.086 \, b_n ((R/R_e)^{1/n} - 1)$ on a linear grid.
4. Make a two-panel plot: log-log ($\Sigma/\Sigma_e$ vs $R/R_e$) and linear ($\mu - \mu_e$ vs $R/R_e$).

In [ ]:
sersic_n = [0.5, 1, 2, 4, 10]

x_min_lin = 0.0
x_max_lin = 5.0
x_min_log = -2.0
x_max_log = 1.7

r_re_lin = np.linspace(x_min_lin, x_max_lin, 200)
r_re_log = 10**np.linspace(x_min_log, x_max_log, 100)
cmap = plt.get_cmap('viridis')

fig, axes = plt.subplots(1, 2, figsize = (9.5, 4.0))
plt.subplots_adjust(wspace = 0.3)

plt.sca(axes[0])
plt.loglog()
plt.xlabel(r'$R/R_{\rm e}$')
plt.ylabel(r'$\Sigma/\Sigma_{\rm e}$')
plt.xlim(10**x_min_log, 10**x_max_log)
plt.ylim(1E-4, 2E3)

plt.sca(axes[1])
plt.xlabel(r'$R/R_{\rm e}$')
plt.ylabel(r'$\mu - \mu_{\rm e}$')
plt.xlim(x_min_lin, x_max_lin)
plt.ylim(8.0, -8.0)

for i in range(len(sersic_n)):
    n = sersic_n[i]
    bn = ... # FILL IN
    sigma = ... # FILL IN
    mu = ... # FILL IN
    
    label = r'$n=%s$' % (str(n))
    if n == 1:
        label += r'$\ \mathrm{(exponential)}$'
    elif n == 4:
        label += r'$\ \mathrm{(de\ Vaucouleurs)}$'
    color = cmap(i / (len(sersic_n) - 1))
    
    plt.sca(axes[0])
    plt.plot(r_re_log, sigma, '-', color = color, label = label)
    plt.sca(axes[1])
    plt.plot(r_re_lin, mu, '-', color = color, label = label)

plt.sca(axes[1])
lg = plt.legend(loc = 1, labelspacing = 0.3)
plt.setp(lg.get_texts(), fontsize = 11)
plt.show()

<details>
<summary>Solution</summary>

```python
    bn = 1.9992 * n - 0.3271
    sigma = np.exp(-bn * (r_re_log**(1.0 / n) - 1.0))
    mu = 1.086 * bn * (r_re_lin**(1.0 / n) - 1.0)
```

Higher $n$ produces more centrally concentrated profiles. The $n=4$ de Vaucouleurs profile drops steeply compared to the $n=1$ exponential. In the $\mu$ plot, the de Vaucouleurs profile appears as a straight line in $R^{1/4}$ space (by construction).

</details>

### SDSS Profile Fits

The SDSS pipeline fits both exponential ($n=1$) and de Vaucouleurs ($n=4$) profiles to every galaxy. The ``modelMag`` uses whichever fit is better.

In [ ]:
# Figure out which galaxies are better fit by exp and which by deV
diff_exp = np.abs(data['modelMag_r'] - data['expMag_r'])
diff_dev = np.abs(data['modelMag_r'] - data['deVMag_r'])
mask_exp = (diff_exp < diff_dev)
re_best = np.array(data['deVRad_r'])
re_best[mask_exp] = data['expRad_r'][mask_exp]

print(f"Fraction best fit by exponential: {np.mean(mask_exp):.2f}")
print(f"Fraction best fit by de Vaucouleurs: {1-np.mean(mask_exp):.2f}")

### Observed Radial Profiles

The SDSS pipeline measures azimuthally-averaged surface brightness profiles in circular annuli for each galaxy. Let's load these profiles and see how well the theoretical predictions hold: exponential galaxies should appear linear in $R$-$\mu$ space, while de Vaucouleurs galaxies should appear linear in $R^{1/4}$-$\mu$ space. Profiles are shown in all five SDSS bands ($u, g, r, i, z$).

In [ ]:
# Load SDSS surface brightness profiles
prf_ids, prfs, prf_err = routines.loadSDSSProfiles()
bin_data = np.loadtxt(os.path.join(routines.data_dir, 'sdss/sdss_profilebins_dr8.txt'),
                      unpack = True, skiprows = 1, delimiter = ',')
bin_lo = bin_data[3]
bin_hi = bin_data[4]
bin_centers_prf = (bin_lo + bin_hi) * 0.5

# Select nearby, luminous, not-too-elongated galaxies that have profiles
np.random.seed(2025)
mask_prf = np.in1d(data['objID'], prf_ids)
mask_prf &= (data['dL'] >= 20.0) & (data['dL'] <= 60.0) & (data['M_cmodel_r'] < -19.0)
mask_prf &= (data['ab_best'] > 0.4)
n_cols_prf = 4

for k, morph_label in enumerate(['Exponential (disk) galaxies', 'de Vaucouleurs (elliptical) galaxies']):
    if k == 0:
        mask_use = mask_prf & data['mask_exp']
    else:
        mask_use = mask_prf & np.logical_not(data['mask_exp'])
    idxs_all = np.where(mask_use)[0]
    idxs_prf = idxs_all[np.random.randint(0, len(idxs_all) - 1, n_cols_prf)]
    
    print(morph_label)
    
    # Plot profiles in linear R and R^(1/4) space
    for l, xlabel_prf in enumerate([r'$R\ ({\rm arcsec})$', r'$R^{1/4}\ ({\rm arcsec}^{1/4})$']):
        fig, axs = plt.subplots(1, n_cols_prf, figsize = (0.5 + n_cols_prf * 2.5, 2.8))
        plt.subplots_adjust(wspace = 0.05, bottom = 0.3)
        for i in range(n_cols_prf):
            plt.sca(axs[i])
            plt.xlabel(xlabel_prf)
            if i == 0:
                plt.ylabel(r'$\mu\ ({\rm mag} / {\rm arcsec}^2)$', labelpad = 10)
            else:
                axs[i].set_yticklabels([])

            # Find matching profile by object ID
            idx_prf = np.where(prf_ids == data['objID'][idxs_prf[i]])[0][0]
            max_R = 0.0
            for j in range(5):
                mask_bins = (prfs[idx_prf, j, :] >= 0.0)
                if l == 0:
                    x = bin_centers_prf[mask_bins]
                else:
                    x = bin_centers_prf[mask_bins]**0.25
                mu_prf = 22.5 - 2.5 * np.log10(prfs[idx_prf, j, mask_bins])
                plt.plot(x, mu_prf, marker = 'o', ms = 2.0, lw = 0.5, c = routines.filter_colors[j])
                max_R = max(max_R, x[-1])
            if l == 0:
                plt.xlim(0.0, max_R * 1.04)
            else:
                plt.xlim(0.5, max_R * 1.04)
            plt.ylim(15.0, 30.0)
            axs[i].invert_yaxis()
        plt.show()

### Galaxy Sizes

The effective radius $R_e$ (half-light radius) varies enormously across the galaxy population. Let's look at the distribution in physical units (kpc).

In [ ]:
dA = cosmo.angularDiameterDistance(data['z']) / cosmo.h * 1000.0
kpc_factor = dA * np.pi / 180.0 / 3600.0
re_best_kpc = re_best * kpc_factor

hist_args = dict(bins = 30, range = [-0.5, 1.6], histtype = 'step')
hist_args_best = hist_args.copy()
hist_args_best.update(histtype = 'stepfilled')

plt.figure()
plt.xlabel(r'$\log_{10} R_{50\%}\ ({\rm kpc})$')
plt.ylabel(r'$N$')
plt.xlim(-0.5, 1.6)
plt.hist(np.log10(re_best_kpc), label = r'$\mathrm{Best\ fit}$', **hist_args_best)
plt.hist(np.log10(data['expRad_r'] * kpc_factor), label = r'$\mathrm{Exponential}$', **hist_args)
plt.hist(np.log10(data['deVRad_r'] * kpc_factor), label = r'$\mathrm{de\ Vaucouleurs}$', **hist_args)
plt.legend()
plt.show()

### Low Surface Brightness (LSB) Galaxies

Not all galaxies are readily detectable. Some spread their light over such a large area that their surface brightness falls below the sky noise, even though their total luminosity may be substantial. These **low surface brightness (LSB) galaxies** are systematically missed by magnitude-limited surveys like SDSS. The mean surface brightness within the half-light radius is

$$\langle \mu \rangle_{50} = m + 2.5 \log_{10}(2\pi R_e^2)$$

where $m$ is the apparent magnitude and $R_e$ is in arcseconds. Galaxies with $\langle \mu \rangle_{50} \gtrsim 23\ \mathrm{mag/arcsec}^2$ are considered LSB. Recently, extremely diffuse objects called **ultra-diffuse galaxies (UDGs)** have been discovered in galaxy clusters, with $R_e > 1.5\ \mathrm{kpc}$ but luminosities comparable to dwarf galaxies.

In [ ]:
# Compute mean surface brightness within Re
np.random.seed(2025)
sb = data['m_petro_r'] + 2.5 * np.log10(2.0 * np.pi * data['Re_best']**2)

# Select luminous, not-too-elongated LSB galaxies at moderate distance
mask_lsb = (data['dL'] >= 100.0) & (data['dL'] <= 300.0)
mask_lsb &= (sb > 23.0)
mask_lsb &= (data['M_petro_r'] < -20.5)
mask_lsb &= (data['expAB_r'] > 0.5)

idxs_lsb = np.where(mask_lsb)[0]
n_lsb_show = min(3 * 4, len(idxs_lsb))
idxs_lsb_sample = idxs_lsb[np.random.randint(0, len(idxs_lsb) - 1, n_lsb_show)]

print(f"Found {len(idxs_lsb)} LSB galaxies with mu > 23 mag/arcsec^2 and M_r < -20.5")
routines.imageCollage(data[idxs_lsb_sample], 3, 4, n_pix = 140, panel_size = 2.0);

### Size-Magnitude Relation

More luminous galaxies are larger. This is a fundamental scaling relation.

In [ ]:
cmap_hist = plt.get_cmap('Blues')
cmap_hist.set_under('#FFFFFF')

mag_lo = -14.0
mag_hi = -25.0
Re_lo = -1.0
Re_hi = 2.2

log_re = np.log10(data['Re_best_kpc'])
M = data['M_cmodel_r']
mask_exp_data = data['mask_exp']
mask_dev = np.logical_not(data['mask_exp'])

hist, _, _ = np.histogram2d(M, log_re, bins = (30, 30), range = [[mag_hi, mag_lo], [Re_lo, Re_hi]])
hist = np.log10(hist.T[::-1] + 1.0)

med_args = dict(statistic = 'median', range = (mag_hi, mag_lo), bins = 20)
median_all, bin_edges, _ = scipy.stats.binned_statistic(M, log_re, **med_args)
median_exp, bin_edges, _ = scipy.stats.binned_statistic(M[mask_exp_data], log_re[mask_exp_data], **med_args)
median_dev, bin_edges, _ = scipy.stats.binned_statistic(M[mask_dev], log_re[mask_dev], **med_args)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) * 0.5

print('Median size varies from %.1f to %.0f kpc.' % (10**median_all[-1], 10**median_all[0]))

plt.figure(figsize = (5.0, 4.0))
plt.xlabel(r'$M_{\rm cModel,r}$')
plt.ylabel(r'$\log_{10} R_{\rm e}\ ({\rm kpc})$')
plt.imshow(hist, extent = [mag_hi, mag_lo, Re_lo, Re_hi], interpolation = 'nearest', aspect = 'auto',
           cmap = cmap_hist, vmin = 0.001, vmax = np.max(hist))
plt.plot(bin_centers, median_all, ls = '-', lw = 1.5, color = routines.color_cycle[1], label = r'$\mathrm{all}$')
plt.plot(bin_centers, median_exp, ls = '--', lw = 0.8, color = routines.color_cycle[1], label = r'$\mathrm{exp}$')
plt.plot(bin_centers, median_dev, ls = '-.', lw = 0.8, color = routines.color_cycle[1], label = r'$\mathrm{deV}$')

x_mw = [routines.Mr_MW]
xerr_mw = [[routines.Mr_MW - routines.Mr_MW_lo], [routines.Mr_MW_hi - routines.Mr_MW]]
log_Re_MW = np.log10(routines.Re_MW)
y_mw = [log_Re_MW]
yerr_mw = [[log_Re_MW - np.log10(routines.Re_MW_lo)], [np.log10(routines.Re_MW_hi) - log_Re_MW]]
plt.errorbar(x_mw, y_mw, xerr = xerr_mw, yerr = yerr_mw, fmt = 's', ms = 4.0, color = routines.color_cycle[3], 
             ecolor = routines.color_cycle[3], lw = 1.0, label = r'$\mathrm{MW}$')

plt.xlim(mag_lo, mag_hi)
plt.ylim(Re_lo, Re_hi)
plt.legend()
cbar = plt.colorbar()
cbar.set_label(r'$\log_{10} (N + 1)$')
plt.show()

# The Galaxy Luminosity Function

## Malmquist Bias

A magnitude-limited survey is biased: brighter galaxies are visible to larger distances and are overrepresented. This is the Malmquist bias.

In [ ]:
z_lo = 0.0
z_hi = 0.4
m_lo = 18.5
m_hi = 12.0

M_std = -23.5
m_lim = routines.m_r_limit
gr_std = 0.8

cmap_mb = plt.get_cmap('Blues')
cmap_mb.set_under('#FFFFFF')
cbar_limit = 3.8

z_array = np.linspace(0.0001, z_hi, 50)
DM = routines.cosmo.distanceModulus(z_array)
K = routines.kCorrection('r', z_array, 'g-r', gr_std)
evo = -1.3 * z_array

fig, axs = plt.subplots(1, 2, figsize = (10.0, 4.0), width_ratios = [1.0, 1.2])
plt.subplots_adjust(wspace = 0.25)

for i in range(2):
    if i == 0:
        y = data['m_petro_r']
        ylabel = r'$m_{\rm petro,r}$'
        y_lo = m_lo
        y_hi = m_hi
    else:
        y = data['M_petro_r']
        ylabel = r'$M_{\rm petro,r}$'
        max_dm = routines.cosmo.distanceModulus(z_hi) - 5
        y_lo = m_lo - max_dm
        y_hi = m_hi - max_dm

    hist_mb, _, _ = np.histogram2d(data['z'], y, bins = (50, 50), range = [[z_lo, z_hi], [y_hi, y_lo]])
    hist_mb = np.log10(hist_mb.T[::-1] + 1.0)

    plt.sca(axs[i])
    plt.xlabel(r'$z$')
    plt.ylabel(ylabel, labelpad = 10)
    plt.imshow(hist_mb, extent = [z_lo, z_hi, y_hi, y_lo], interpolation = 'nearest', aspect = 'auto',
               cmap = cmap_mb, vmin = 0.001, vmax = cbar_limit)
    c_lines = routines.color_cycle[1]
    if i == 0:
        m_std_val = M_std + DM
        plt.plot(z_array, m_std_val + K, ':', c = c_lines, lw = 0.8, label = r'$m_{\rm r} = -23.5 + {\rm DM} + K$')
        plt.plot(z_array, np.ones_like(z_array) * m_lim, '--', c = c_lines, lw = 1.2, label = r'$m_{\rm r} = 17.77$')
    else:
        M_lim = m_lim - DM - K
        plt.plot(z_array, M_lim, '--', c = c_lines, lw = 1.2, label = r'$M_{\rm r} = 17.77 - {\rm DM} - K$')
    plt.xlim(z_lo, z_hi)
    plt.ylim(y_lo, y_hi)
    legend_args = dict(labelspacing = 0.2, handlelength = 1.5, borderpad = 0.0)
    if i == 0:
        lg = plt.legend(**legend_args)
    else:
        lg = plt.legend(loc = 4, **legend_args)
        cbar = plt.colorbar()
        cbar.set_label(r'$\log_{10} (N + 1)$')
    plt.setp(lg.get_texts(), fontsize = 10)
plt.show()

### The $V_{\rm max}$ Method

To correct for Malmquist bias, we weight each galaxy by $1/V_{\rm max}$, the inverse of the maximum volume within which it could have been observed given the survey's magnitude limit.

In [ ]:
z_min = 0.02
z_max = None
m_max = routines.m_r_limit
M_petro = data['M_petro_r']

def distanceFromMagLimit(m_limit, M_galaxies):
    dL_lim = 10**(-5.0 + 0.2 * (m_limit - M_galaxies))
    z_lim = routines.cosmo.luminosityDistance(dL_lim * routines.cosmo.h, inverse = True)
    if z_max is not None:
        z_lim = np.minimum(z_lim, z_max)
        dL_lim = routines.cosmo.luminosityDistance(z_lim)
    dC = dL_lim / (1.0 + z_lim)
    return dC

dC_max = distanceFromMagLimit(m_max, M_petro)
Vmax = routines.solid_angle / 3.0 * dC_max**3 / routines.spectroscopic_completeness
Vmax_inv_all = 1.0 / Vmax

# Cut very nearby galaxies to avoid extreme corrections
mask_vmax = (data['z'] >= z_min)
if z_max is not None:
    mask_vmax &= (data['z'] <= z_max)
data_vmax = data[mask_vmax]
Vmax_inv = Vmax_inv_all[mask_vmax]
norm = np.sum(Vmax_inv)
Vmax_inv_norm = Vmax_inv / norm

print(f"Using {len(data_vmax)} galaxies after z > {z_min} cut")

### Corrected Magnitude and Color Distributions

The $V_{\rm max}$ correction dramatically changes the magnitude and color distributions. The corrected color distribution reveals a clear bimodality between blue and red galaxies.

In [ ]:
M_vmax = data_vmax['M_petro_r']
hist_args = dict(range = [-24.0, -16], bins = 30, density = True, log = False)

fig, axes = plt.subplots(1, 2, figsize = (9, 4))

plt.sca(axes[0])
plt.gca().invert_xaxis()
plt.xlabel(r'$M_{\rm petro,r}$')
plt.ylabel(r'${\rm d} N_{\rm gal} / N_{\rm gal} / {\rm d} M$')
plt.hist(M_vmax, histtype = 'stepfilled', label = r'$\mathrm{Corrected}$', weights = Vmax_inv_norm, **hist_args)
plt.hist(M_vmax, histtype = 'step', label = r'$\mathrm{Uncorrected}$', **hist_args)
plt.legend()

gr_vmax = data_vmax['color_gr']
hist_args2 = dict(bins = 30, range = [0.1, 1.1], density = True, log = False)

plt.sca(axes[1])
plt.xlabel(r'$g-r$')
plt.ylabel(r'${\rm d} N_{\rm gal} / N_{\rm gal} / {\rm d} (g-r)$')
plt.hist(gr_vmax, histtype = 'stepfilled', label = r'$\mathrm{Corrected}$', weights = Vmax_inv_norm, **hist_args2)
plt.hist(gr_vmax, histtype = 'step', label = r'$\mathrm{Uncorrected}$', **hist_args2)
plt.legend()

plt.tight_layout()
plt.show()

## In-Class Exercise: Schechter Luminosity Function

### Goal
Construct the galaxy luminosity function from the SDSS data using the $V_{\rm max}$ method and fit a Schechter function.

### Background
The Schechter luminosity function is

$$\phi(L) \, d\log L = \ln(10) \, \phi^* \left(\frac{L}{L^*}\right)^{\alpha+1} \exp\left(-\frac{L}{L^*}\right) d\log L$$

### Task

1. Convert absolute magnitudes to luminosities using $\log_{10}(L/L_\odot) = 0.4 \times (M_{\odot,r} - M_r)$.
2. Build the luminosity function by histogramming $\log L$ weighted by $1/V_{\rm max}$.
3. Define a Schechter function and fit it to the data.
4. Plot both the corrected and uncorrected luminosity functions, plus the fit.

In [ ]:
def schechterFunction(log_L, phi_star, log_Lstar, alpha):
    LLstar = 10**log_L / 10**log_Lstar
    lf = np.log(10.0) * phi_star * LLstar**(alpha + 1.0) * np.exp(-LLstar)
    return np.log10(lf)

# Compute luminosity in solar units with evolution correction
M_sch = data_vmax['M_cmodel_r'] + 1.3 * data_vmax['z']
log_L = ... # FILL IN

# Create luminosity function bins
bin_edges = np.linspace(8.8, 11.7, 50)
bin_centers_lf = 0.5 * (bin_edges[:-1] + bin_edges[1:])
dL_bin = bin_edges[1:] - bin_edges[:-1]

# Uncorrected LF (using total survey volume)
z_min_sdss = 0.001
z_max_sdss = 0.6
dC_max_sdss = routines.cosmo.comovingDistance(0.0, z_max_sdss)
dC_min_sdss = routines.cosmo.comovingDistance(0.0, z_min_sdss)
Vsdss = routines.solid_angle / 3.0 * (dC_max_sdss**3 - dC_min_sdss**3) / routines.spectroscopic_completeness
lf_biased, _ = np.histogram(log_L, bins = bin_edges)
lf_biased = lf_biased / dL_bin / Vsdss

# Vmax-corrected LF
lf_vmax, _ = ... # FILL IN
lf_vmax = lf_vmax / dL_bin

# Fit Schechter function
initial_guess = [2.5E-3, 10.5, -1.0]
params, _ = ... # FILL IN
fit = 10**schechterFunction(bin_centers_lf, params[0], params[1], params[2])
print('Best fit Schechter parameters:')
print('phi*  = %.2e Mpc^-3 dex^-1' % params[0])
print('L*    = 10^%.2f Lsun' % params[1])
print('alpha = %.2f' % params[2])

# Compare to Blanton et al. 2003 (ApJ, 592, 819) SDSS r-band Schechter fit
h = cosmo.h
phi_star_b = 1.49E-2 * h**3
Mstar_b = -20.44 + 5.0 * np.log10(h)
log_Lstar_b = 0.4 * (routines.solar_mag['r'] - Mstar_b)
alpha_b = -1.05
fit_blanton = 10**schechterFunction(bin_centers_lf, phi_star_b, log_Lstar_b, alpha_b)

# Plot
plt.figure(figsize = (3.5, 3.5))
plt.xlabel(r'$L_{\rm r}\ (L_\odot)$')
plt.ylabel(r'$\phi\ ({\rm dex}^{-1} {\rm Mpc}^{-3})$')
plt.loglog()
plt.plot(10**bin_centers_lf, lf_vmax, '-', label = r'$\mathrm{Corrected}$')
plt.plot(10**bin_centers_lf, lf_biased, '-', label = r'$\mathrm{Uncorrected}$')
plt.plot(10**bin_centers_lf, fit, '--', color = 'gray', label = r'$\mathrm{Schechter\ fit}$')
plt.plot(10**bin_centers_lf, fit_blanton, ':', color = 'gray', label = r'$\mathrm{Blanton+03}\ z = 0.1$')
plt.ylim(1E-6, 4E-2)
plt.legend(frameon = True)
plt.show()

<details>
<summary>Solution</summary>

```python
log_L = 0.4 * (routines.solar_mag['r'] - M_sch)
lf_vmax, _ = np.histogram(log_L, bins = bin_edges, weights = Vmax_inv)
params, _ = scipy.optimize.curve_fit(schechterFunction, bin_centers_lf, np.log10(lf_vmax), p0 = initial_guess)
```

The Schechter function fits the corrected luminosity function well. The key parameters are:
- $\phi^* \sim 10^{-3}$ Mpc$^{-3}$ dex$^{-1}$ (normalization)
- $L^* \sim 10^{10.5} L_\odot$ (characteristic luminosity, the "knee")  
- $\alpha \sim -1$ (faint-end slope)

The uncorrected LF drops off at low luminosities due to Malmquist bias, while the corrected version shows the steep faint-end rise. The dotted line shows the "official" SDSS $r$-band Schechter fit from [Blanton et al. 2003](https://ui.adsabs.harvard.edu/abs/2003ApJ...592..819B), which uses a more sophisticated methodology but agrees reasonably well with our simple $V_{\rm max}$ result.

</details>

# Galaxy Morphology

## The Color-Magnitude Diagram

Galaxy morphology correlates strongly with color: ellipticals tend to be red and concentrated, while spirals tend to be blue with lower concentration. The color bimodality is one of the most striking features of the galaxy population.

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib as mpl

mask_images = (data['dL'] < 150.0)
mask_sample = (data['z'] > 0.02)

var_M = 'M_cmodel_r'
label_M = r'$M_{\rm cModel,r}$'
M_lo = -23.5
M_hi = -17.0

var_gr = 'color_gr'
label_gr = r'$g-r$'
gr_lo = 0.0
gr_hi = 1.0

cmap_cmd = plt.get_cmap('Blues')
cmap_cmd.set_under('#FFFFFF')
vmin = 1E-3

data_use = data[mask_sample]
M_cmd = data_use[var_M]
gr_cmd = data_use[var_gr]

hist_cmd, _, _ = np.histogram2d(M_cmd, gr_cmd, bins = (40, 40), range = [[M_lo, M_hi], [gr_lo, gr_hi]], 
                                weights = data_use['1/Vmax'], density = True)
hist_cmd = np.log10(hist_cmd.T[::-1] + 1.0)
vmax_cmd = np.max(hist_cmd)

fig = plt.figure(figsize = (5.5, 4.5))
gs = gridspec.GridSpec(2, 4, height_ratios = [0.2, 1.0], width_ratios = [1.0, 0.2, 0.05, 0.2])
plt.subplots_adjust(left = 0.14, right = 0.96, top = 0.98, bottom = 0.14, hspace = 0.06, wspace = 0.09)
p_main = fig.add_subplot(gs[4])
p_hx = fig.add_subplot(gs[0], sharex = p_main)
p_hy = fig.add_subplot(gs[5], sharey = p_main)
p_cbar = fig.add_subplot(gs[6])

plt.sca(p_main)
plt.xlabel(label_M)
plt.ylabel(label_gr)
plt.xlim(M_lo, M_hi)
plt.ylim(gr_lo, gr_hi)
plt.gca().invert_xaxis()
plt.imshow(hist_cmd, extent = [M_lo, M_hi, gr_lo, gr_hi], interpolation = 'nearest', aspect = 'auto',
           cmap = cmap_cmd, vmin = vmin, vmax = vmax_cmd)

plt.sca(p_hx)
plt.hist(M_cmd, bins = 40, range = [M_lo, M_hi], weights = data_use['1/Vmax'], density = True, histtype = 'step')
p_hx.tick_params(axis = 'x', labelbottom = False)
p_hx.tick_params(axis = 'y', labelleft = False)

plt.sca(p_hy)
p_hy.tick_params(axis = 'y', labelleft = False)
p_hy.tick_params(axis = 'x', labelbottom = False)
plt.hist(gr_cmd, bins = 40, range = [gr_lo, gr_hi], weights = data_use['1/Vmax'], density = True, histtype = 'step', orientation='horizontal')

plt.sca(p_cbar)
cb = mpl.colorbar.ColorbarBase(p_cbar, orientation = 'vertical', cmap = cmap_cmd, norm = mpl.colors.Normalize(vmin = vmin, vmax = vmax_cmd))
cb.set_label(r'$\log_{10} ({\rm d} N / N / {\rm d} x / {\rm d} y + 1)$', rotation = 90, labelpad = 10)

plt.show()

### Morphology and Color

Galaxy images sorted by magnitude and color reveal the morphological trends: red, concentrated ellipticals vs. blue, extended spirals.

In [ ]:
routines.imageCollageVariables(data, var_M, var_gr, label_M, label_gr, M_lo, M_hi, gr_lo, gr_hi, 
                    invert_x = True, n_bins_x = 10, n_bins_y = 10, 
                    mask_images = mask_images, image_size_kpc = 25.0,
                    plot_contours = True, mask_contours = mask_sample, 
                    save = False, show = True);

### Concentration

Concentration $c = R_{90\%}/R_{50\%}$ measures how centrally concentrated a galaxy's light is. It separates spirals (low $c$) from ellipticals (high $c$) better than surface brightness alone.

In [ ]:
var_c = 'c_90_50_r'
label_c = r'$c = R_{90\%} / R_{50\%}$'
c_lo = 1.5
c_hi = 3.6

routines.imageCollageVariables(data, var_M, var_c, label_M, label_c, M_lo, M_hi, c_lo, c_hi, 
                    invert_x = True, n_bins_x = 10, n_bins_y = 10, 
                    mask_images = mask_images, image_size_kpc = 25.0,
                    plot_contours = True, mask_contours = mask_sample,
                    save = False, show = True);

### Concentration and Color

The bimodality in color persists in the concentration-color plane: elliptical galaxies tend to be BOTH red and concentrated, while spirals tend to be BOTH blue and low-concentration.

In [ ]:
mask_im_m19 = (mask_images & (data['M_cmodel_r'] < -19.0))

routines.imageCollageVariables(data, var_c, var_gr, label_c, label_gr, c_lo, c_hi, gr_lo, gr_hi, 
                    n_bins_x = 10, n_bins_y = 10, 
                    mask_images = mask_im_m19, image_size_kpc = 25.0,
                    plot_contours = True, mask_contours = mask_sample,
                    save = False, show = True);

### Axis Ratio

Edge-on disk galaxies can mimic high-concentration ellipticals because the line-of-sight path through the disk is long, making the central surface brightness appear higher. The axis ratio $b/a$ (minor-to-major axis from the best-fit profile) helps break this degeneracy: edge-on spirals have low $b/a$ while true ellipticals tend to have $b/a \gtrsim 0.5$. A cut on high concentration *and* high $b/a$ isolates genuine ellipticals more cleanly.

In [ ]:
var_ab = 'ab_best'
label_ab = r'$b/a$'
ab_lo = 0.0
ab_hi = 1.0

routines.imageCollageVariables(data, var_c, var_ab, label_c, label_ab, c_lo, c_hi, ab_lo, ab_hi,
                    n_bins_x = 10, n_bins_y = 10,
                    mask_images = mask_im_m19, image_size_kpc = 25.0,
                    plot_contours = True, mask_contours = mask_sample,
                    save = False, show = True);

### Machine-Learning Morphological Classifications

The simple diagnostics above (color, concentration, axis ratio) provide crude morphological separation but are far from perfect. [Huertas-Company et al. (2011)](https://ui.adsabs.harvard.edu/abs/2011A%26A...525A.157H) applied machine-learning techniques to the SDSS sample, providing probabilistic classifications into four Hubble types: elliptical (E), lenticular (S0), early spiral (Sab), and late spiral (Scd). These probabilities are included in the UPenn photometric catalog of [Meert et al. (2015)](https://ui.adsabs.harvard.edu/abs/2015MNRAS.446.3943M). Let's visualize galaxies that are confidently classified into each type.

In [ ]:
# Load UPenn catalog with Huertas-Company morphological probabilities
d_upenn = routines.loadUPennCatalog()
print(f"UPenn catalog: {len(d_upenn)} galaxies")

n_rows_morph = 5
n_cols_morph = 5
n_pix_morph = 120
min_prob = 0.7
np.random.seed(2025)

dL_upenn = cosmo.luminosityDistance(d_upenn['z']) / cosmo.h
mask_upenn = (dL_upenn >= 50.0) & (dL_upenn <= 150.0)
mask_upenn &= (d_upenn['m_tot_r'] < 16.0)

p_Ell = d_upenn['probaEll_h11']
p_S0 = d_upenn['probaS0_h11']
p_Sab = d_upenn['probaSab_h11']
p_Scd = d_upenn['probaScd_h11']

morph_labels = ['Elliptical (E)', 'Lenticular (S0)', 'Early Spiral (Sab)', 'Late Spiral (Scd)']
morph_probs = [p_Ell, p_S0, p_Sab, p_Scd]
all_probs = [p_Ell, p_S0, p_Sab, p_Scd]

for i in range(4):
    p = morph_probs[i]
    # Require this type to have the highest probability and exceed threshold
    mask_morph = mask_upenn & (p > min_prob)
    for j in range(4):
        if j != i:
            mask_morph &= (p > all_probs[j])
    
    n_found = np.count_nonzero(mask_morph)
    if n_found == 0:
        continue
    print(f"{morph_labels[i]}: {n_found} galaxies with p > {min_prob}")
    
    idxs_valid = np.where(mask_morph)[0]
    idxs_show = idxs_valid[np.random.randint(0, len(idxs_valid) - 1, n_rows_morph * n_cols_morph)]
    routines.imageCollage(d_upenn[idxs_show], n_rows_morph, n_cols_morph, 
                         n_pix = n_pix_morph, panel_size = 1.5);

## The Milky Way Rotation Curve

The rotation curve measures circular velocity as a function of radius. For the Milky Way, the disk + bulge model matches observations at small radii but falls far short at large radii. This is one of the strongest motivations for dark matter.

In [ ]:
from colossus.utils import constants
from colossus.halo import profile_hernquist

# Disk data from Licquia & Newman 2016, bulge from Ninkovic 2017
M_disk = 4.8E10
R_d = routines.Rd_MW
M_bulge = 1.85E10
R_bulge = 0.3

# Load rotation curve data
d_rc = np.loadtxt(os.path.join(routines.data_dir, 'milky_way/rotation_curve_sofue_2020.txt'), unpack = True)

# Exponential disk (Binney & Tremaine eq. 2.165)
Sigma_0 = M_disk / (2.0 * np.pi * R_d**2)
R = np.linspace(1E-5, 100.0, 200)
y = R / 2.0 / R_d
I_0 = scipy.special.iv(0, y)
I_1 = scipy.special.iv(1, y)
K_0 = scipy.special.kn(0, y)
K_1 = scipy.special.kn(1, y)
V_disk = np.sqrt(4.0 * np.pi * constants.G * Sigma_0 * R_d * y**2 * (I_0 * K_0 - I_1 * K_1))

# Hernquist bulge
h = routines.cosmo.h
prf = profile_hernquist.HernquistProfile(rhos = 1.0, rs = R_bulge * h)
M_norm = prf.enclosedMass(1000.0) / h
prf = profile_hernquist.HernquistProfile(rhos = M_bulge / M_norm, rs = R_bulge * h)
V_bulge = prf.circularVelocity(R * h)

V_tot = np.sqrt(V_bulge**2 + V_disk**2)

plt.figure(figsize = (6.0, 4.0))
plt.xlabel(r'$R\ ({\rm kpc})$')
plt.ylabel(r'$v_{\rm c}\ ({\rm km} / {\rm s})$')
plt.axvline(8.0, ls = '--', color = 'gray', lw = 0.8, label = r'$\mathrm{Solar\ position}$')
plt.errorbar(d_rc[0], d_rc[1], d_rc[2], fmt = 'o', ms = 2.0, color = 'gray', lw = 0.5, label = r'$\mathrm{Observations}$')
plt.plot(R, V_tot, '-', label = r'$\mathrm{Total}$', lw = 0.9, zorder = 100)
plt.plot(R, V_disk, '-.', label = r'$\mathrm{Exponential\ disk}$', lw = 0.8, color = routines.color_cycle[1])
plt.plot(R, V_bulge, '--', label = r'$\mathrm{Bulge}$', lw = 0.8, color = routines.color_cycle[1])
plt.xlim(-2.0, 100.0)
plt.ylim(0.0, 290.0)
plt.legend(labelspacing = 0.1, borderpad = 0.1)
plt.show()